In [119]:
from google import genai
from google.genai import types
import wave
from loguru import logger
from pathlib import Path
import json

import asyncio
import json
import wave
from pathlib import Path
from datetime import datetime
from difflib import SequenceMatcher

import numpy as np
import soundfile as sf
from jiwer import wer

from loguru import logger
from pydantic import BaseModel

from dotenv import load_dotenv
import os

load_dotenv()

True

In [73]:
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

# load the metadata

In [5]:

# load the data 
BASE_DIR = Path("__file__").resolve().parent
BASE_DIR
DATA_PATH = Path(r"..\data\synthetic_audio_dataset.jsonl")



In [6]:

audio_data = []

with open(DATA_PATH, "r" , encoding="utf-8") as f:
    for line in f:
        audio_data.append(json.loads(line))

print(audio_data)

[{'audio_id': '6d383aa7-5c62-400c-8113-caba39a71aeb', 'prompt_id': '6d383aa7-5c62-400c-8113-caba39a71aeb', 'audio_path': 'data\\audio_outputs\\6d383aa7-5c62-400c-8113-caba39a71aeb.wav', 'text': 'ايه الاخبار يا باشا؟ عامل ايه النهاردة؟', 'voice_name': 'Algenib', 'speaker_id': '3', 'tts_model': 'gemini-2.5-flash-preview-tts', 'background_noise': 'clean audio', 'sample_rate': 24000, 'status': 'generated', 'review_status': 'pending'}, {'audio_id': '759c7a5b-78a7-4749-aeb1-6122cb88b9ec', 'prompt_id': '759c7a5b-78a7-4749-aeb1-6122cb88b9ec', 'audio_path': 'data\\audio_outputs\\759c7a5b-78a7-4749-aeb1-6122cb88b9ec.wav', 'text': 'لو سمحت، هو البنطلون ده بكام؟', 'voice_name': 'Algenib', 'speaker_id': '3', 'tts_model': 'gemini-2.5-flash-preview-tts', 'background_noise': 'background street noise', 'sample_rate': 24000, 'status': 'generated', 'review_status': 'pending'}, {'audio_id': 'ef2b9043-9163-4aa8-8289-8ff1e22ce3f5', 'prompt_id': 'ef2b9043-9163-4aa8-8289-8ff1e22ce3f5', 'audio_path': 'data\\au

In [7]:
audio_data[0]

{'audio_id': '6d383aa7-5c62-400c-8113-caba39a71aeb',
 'prompt_id': '6d383aa7-5c62-400c-8113-caba39a71aeb',
 'audio_path': 'data\\audio_outputs\\6d383aa7-5c62-400c-8113-caba39a71aeb.wav',
 'text': 'ايه الاخبار يا باشا؟ عامل ايه النهاردة؟',
 'voice_name': 'Algenib',
 'speaker_id': '3',
 'tts_model': 'gemini-2.5-flash-preview-tts',
 'background_noise': 'clean audio',
 'sample_rate': 24000,
 'status': 'generated',
 'review_status': 'pending'}

# RESULT SCHEMA

In [131]:
class ValidationResult(BaseModel):

    audio_id: str

    status: str = "reviewed"

    review_status: str

    duration_seconds: float

    wer: float | None = None

    transcript: str | None = None

    issues: list[str] = []

    validated_at: str

# AUDIO UTILS

In [169]:
from pathlib import Path

BASE_DIR = Path.cwd().parent


def resolve_audio_path(
    audio_path: str | Path
) -> Path:

    audio_path = Path(audio_path)

    if audio_path.is_absolute():

        return audio_path

    full_path = (BASE_DIR / audio_path).resolve()

    if not full_path.exists():

        raise FileNotFoundError(
            f"Audio file not found: {full_path}"
        )

    return full_path

In [122]:

def get_audio_duration(audio_path: str) -> float:

    with wave.open(audio_path, "rb") as wf:

        frames = wf.getnframes()
        rate = wf.getframerate()

        duration = frames / float(rate)

    return duration


def compute_rms_energy(audio: np.ndarray) -> float:

    return np.sqrt(np.mean(audio ** 2))


def detect_clipping(audio: np.ndarray, threshold=0.99):

    peak = np.max(np.abs(audio))

    return peak >= threshold


def text_similarity(text1: str, text2: str):

    return SequenceMatcher(
        None,
        text1,
        text2
    ).ratio()

def text_WER(text1: str, text2: str):

    return wer(
            text1,
            text2
        )


In [123]:


text_WER(
    "السلام عليكم",
    "السلام عليكم"
) 

0.0

In [116]:
text_similarity("السلام عليكم" , "السلام لك")

0.8571428571428571

# eval stage 1 : Fast Rule Filters

In [61]:
min_duration=1.0
max_duration=30.0

min_rms=0.005
min_similarity_score=0.70

In [68]:
def run_fast_checks(
        
        audio_path: str,
        text: str,
        min_duration=1.0,
        max_duration=30.0,
        min_rms=0.005



    ):

        issues = []

        # =========================================
        # Duration
        # =========================================

        duration = get_audio_duration(audio_path)

        print("duration : ", duration)

        print(min_duration)

        if duration < min_duration:

            issues.append(
                f"audio too short ({duration:.2f}s)"
            )

        if duration > max_duration:

            issues.append(
                f"audio too long ({duration:.2f}s)"
            )

        # =========================================
        # Load audio
        # =========================================

        audio, sr = sf.read(audio_path)

        if audio.ndim > 1:
            audio = audio.mean(axis=1)

        audio = audio.astype(np.float32)

        # =========================================
        # RMS / Silence
        # =========================================

        rms = compute_rms_energy(audio)

        if rms < min_rms:

            issues.append(
                f"low audio energy ({rms:.6f})"
            )

        # =========================================
        # Clipping
        # =========================================

        if detect_clipping(audio):

            issues.append("audio clipping detected")

        # =========================================
        # Empty text
        # =========================================

        if len(text.strip()) == 0:

            issues.append("empty text")

        # =========================================
        # Final
        # =========================================

        passed = len(issues) == 0

        return {

            "passed": passed,

            "issues": issues,

            "duration": duration
        }

In [128]:
audio_path = Path("..") / audio_data[2]['audio_path']

text = audio_data[2]['text']

In [76]:
str(audio_path)

'..\\data\\audio_outputs\\759c7a5b-78a7-4749-aeb1-6122cb88b9ec.wav'

In [171]:
resolve_audio_path("data\\audio_outputs\\759c7a5b-78a7-4749-aeb1-6122cb88b9ec.wav")

WindowsPath('D:/GAN_AI/Synthetic-Speech-Data-Pipeline-For-STT/data/audio_outputs/759c7a5b-78a7-4749-aeb1-6122cb88b9ec.wav')

In [77]:
print(get_audio_duration(str(audio_path)))

2.9309583333333333


In [78]:
print(type(get_audio_duration(str(audio_path))))

<class 'float'>


In [79]:
run_fast_checks(str(audio_path) , text)

duration :  2.9309583333333333
1.0


{'passed': True, 'issues': [], 'duration': 2.9309583333333333}

# eval stage 2 : STT Verification

In [80]:
from google import genai

client = genai.Client(api_key=GEMINI_API_KEY)

myfile = client.files.upload(file=audio_path)

response = client.models.generate_content(
    model="gemini-2.5-flash", contents=["Process the audio file and generate a detailed transcription", myfile]
)

print(response.text)

لو سمحت، هو البنطلون ده بكام؟


In [85]:
async def transcribe_audio(audio_path: str):

    uploaded_file = await asyncio.to_thread(
        client.files.upload,
        file=audio_path
    )

    result = await asyncio.to_thread(
        client.models.generate_content,
        model="gemini-2.5-flash",
        contents=[
            "Process the audio file and generate a detailed transcription",
            uploaded_file,
        ],
    )

    return result.text

In [86]:
await transcribe_audio(audio_path)

'لو سمحت، هو البنطلون ده بكام؟'

In [109]:
import re

def normalize_arabic(text: str):

    text = text.lower()

    text = re.sub(r'[ًٌٍَُِّْـ]', '', text)

    text = re.sub(r'[^\w\s]', '', text)

    text = re.sub(r'\s+', ' ', text).strip()

    return text

In [110]:
async def run_stt_verification(
    audio_path: str,
    original_text: str,
    max_wer: float = 0.25
):

    issues = []

    try:

        # =====================================
        # STT
        # =====================================

        transcript = await transcribe_audio(
            audio_path
        )

        transcript = normalize_arabic(
            transcript.strip()
        )

        reference = normalize_arabic(
            original_text.strip()
        )

        # =====================================
        # WER
        # =====================================

        wer_score = text_WER(
            reference,
            transcript
        )

        passed = wer_score <= max_wer

        if not passed:

            issues.append(
                f"high wer ({wer_score:.2f})"
            )

        return {

            "passed": passed,

            "wer": wer_score,

            "transcript": transcript,

            "issues": issues
        }

    except Exception as e:

        logger.exception(
            "STT verification failed"
        )

        return {

            "passed": False,

            "wer": 1.0,

            "transcript": "",

            "issues": [str(e)]
        }

In [129]:
text

'لو سمحت، عايز طبق كشري كبير ومش عايزه سبايسي خالص، وشكرا.'

In [130]:
await run_stt_verification(audio_path , text)

{'passed': True,
 'wer': 0.0,
 'transcript': 'لو سمحت عايز طبق كشري كبير ومش عايزه سبايسي خالص وشكرا',
 'issues': []}

# full class

In [ ]:
class AudioValidationService:

    def __init__(
        self,

        accepted_jsonl="data/accepted.jsonl",
        rejected_jsonl="data/rejected.jsonl",

        min_duration=1.0,
        max_duration=30.0,

        min_rms=0.005,

        max_wer: float = 0.25
    ):


        self.accepted_jsonl = Path(accepted_jsonl)
        self.rejected_jsonl = Path(rejected_jsonl)

        self.accepted_jsonl.parent.mkdir(
            parents=True,
            exist_ok=True
        )

        self.max_wer = max_wer

        self.min_duration = min_duration
        self.max_duration = max_duration
        self.min_rms = min_rms
    # =====================================================
    # SAVE JSONL
    # =====================================================

    def save_result(
            self,
            result: dict,
            accepted: bool
                ):

            output_file = (
                self.accepted_jsonl
                if accepted
                else self.rejected_jsonl
            )

            with open(
                output_file,
                "a",
                encoding="utf-8"
            ) as f:

                f.write(
                    json.dumps(
                        result,
                        ensure_ascii=False
                    ) + "\n"
                )

    # =====================================================
    # STAGE 1 -> FAST RULE FILTERS
    # =====================================================

    def run_fast_checks(
        self,
        audio_path: str,
        text: str
    ):

        return run_fast_checks(audio_path , text , self.min_duration , self.max_duration, self.min_rms  )

    # =====================================================
    # STAGE 2 -> STT VERIFICATION
    # =====================================================

    async def transcribe_audio(
        self,
        audio_path: str
    ):
        return await transcribe_audio(audio_path)


    async def run_stt_verification(
        self,
        audio_path: str,
        original_text: str
    ):

        return await run_stt_verification(audio_path , original_text , self.max_wer)

    # =====================================================
    # FULL VALIDATION PIPELINE
    # =====================================================

    async def validate_sample(
                self,
                record: dict
        ):

            logger.info(
                f"🔍 Validating sample | "
                f"id={record['audio_id']}"
            )

            audio_path = str(resolve_audio_path(record["audio_path"]))
            original_text = record["text"]

            # =====================================
            # STAGE 1
            # =====================================

            fast_checks = self.run_fast_checks(
                audio_path,
                original_text
            )

            print("fast_checks output: ", fast_checks)

            # =====================================
            # REJECT EARLY
            # =====================================

            if not fast_checks["passed"]:

                record.update({

                    "status": "reviewed",

                    "review_status": "rejected",

                    "duration_seconds": fast_checks["duration"],

                    "wer": None,

                    "transcript": None,

                    "issues": fast_checks["issues"],

                    "validated_at": datetime.utcnow().isoformat()
                })

                self.save_result(
                    record,
                    accepted=False
                )

                logger.warning(
                    f"❌ Rejected by stage 1 | "
                    f"id={record['audio_id']}"
                )

                return record

            # =====================================
            # STAGE 2
            # =====================================

            stt_result = await self.run_stt_verification(
                audio_path,
                original_text
            )

            accepted = stt_result["passed"]

            review_status = (
                "accepted"
                if accepted
                else "rejected"
            )

            # =====================================
            # FINAL RECORD
            # =====================================

            record.update({

                "status": "reviewed",

                "review_status": review_status,

                "duration_seconds": fast_checks["duration"],

                "wer": stt_result["wer"],

                "transcript": stt_result["transcript"],

                "issues": (
                    fast_checks["issues"]
                    + stt_result["issues"]
                ),

                "validated_at": datetime.utcnow().isoformat()
            })

            self.save_result(
                record,
                accepted=accepted
            )

            logger.success(
                f"✅ Validation complete | "
                f"id={record['audio_id']} | "
                f"decision={review_status}"
            )

            return record

    # =====================================================
    # VALIDATE MULTIPLE
    # =====================================================

    async def validate_parallel(
        self,
        records: list[dict]
    ):

        logger.info(
            f"🚀 Starting validation batch | "
            f"samples={len(records)}"
        )

        tasks = [
            self.validate_sample(r)
            for r in records
        ]

        results = await asyncio.gather(*tasks)

        logger.success(
            f"🏁 Validation completed | "
            f"samples={len(results)}"
        )

        return results

In [185]:
audio_val =  AudioValidationService()

In [190]:
audio_data[0]

{'audio_id': '6d383aa7-5c62-400c-8113-caba39a71aeb',
 'prompt_id': '6d383aa7-5c62-400c-8113-caba39a71aeb',
 'audio_path': 'data\\audio_outputs\\6d383aa7-5c62-400c-8113-caba39a71aeb.wav',
 'text': 'ايه الاخبار يا باشا؟ عامل ايه النهاردة؟',
 'voice_name': 'Algenib',
 'speaker_id': '3',
 'tts_model': 'gemini-2.5-flash-preview-tts',
 'background_noise': 'clean audio',
 'sample_rate': 24000,
 'status': 'generated',
 'review_status': 'pending'}

In [188]:
await audio_val.validate_sample(audio_data[2])

2026-05-11 21:18:33.711 | INFO     | __main__:validate_sample:105 - 🔍 Validating sample | id=ef2b9043-9163-4aa8-8289-8ff1e22ce3f5


duration :  5.890958333333334
1.0
fast_checks output:  {'passed': True, 'issues': [], 'duration': 5.890958333333334}


C:\Users\DELL\AppData\Local\Temp\ipykernel_11352\3106386381.py:197: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "validated_at": datetime.utcnow().isoformat()
2026-05-11 21:18:37.750 | SUCCESS  | __main__:validate_sample:205 - ✅ Validation complete | id=ef2b9043-9163-4aa8-8289-8ff1e22ce3f5 | decision=accepted


{'audio_id': 'ef2b9043-9163-4aa8-8289-8ff1e22ce3f5',
 'prompt_id': 'ef2b9043-9163-4aa8-8289-8ff1e22ce3f5',
 'audio_path': 'data\\audio_outputs\\ef2b9043-9163-4aa8-8289-8ff1e22ce3f5.wav',
 'text': 'لو سمحت، عايز طبق كشري كبير ومش عايزه سبايسي خالص، وشكرا.',
 'voice_name': 'Algieba',
 'speaker_id': '4',
 'tts_model': 'gemini-2.5-flash-preview-tts',
 'background_noise': 'background crowd',
 'sample_rate': 24000,
 'status': 'reviewed',
 'review_status': 'accepted',
 'duration_seconds': 5.890958333333334,
 'wer': 0.0,
 'transcript': 'لو سمحت عايز طبق كشري كبير ومش عايزه سبايسي خالص وشكرا',
 'issues': [],
 'validated_at': '2026-05-11T18:18:37.749031'}

In [192]:
res = await audio_val.validate_parallel(audio_data[:4])

2026-05-11 21:22:10.294 | INFO     | __main__:validate_parallel:222 - 🚀 Starting validation batch | samples=0
2026-05-11 21:22:10.295 | SUCCESS  | __main__:validate_parallel:234 - 🏁 Validation completed | samples=0
